In [18]:

def load_data():
    import pandas as pd
    return pd.read_csv("data.csv")  # default dataset

df = load_data()


In [19]:
import pandas as pd
import numpy as np

class TargetDetector:
    def __init__(self, problem_type="auto"):
        self.problem_type = problem_type
        
    def detect(self, df):
        scores = {}
        n_rows = len(df)

        for col in df.columns:
            series = df[col]
            unique_ratio = series.nunique() / n_rows
            
            score = 0

            # Penalize ID-like columns
            if unique_ratio > 0.95:
                score -= 2

            # Reward low cardinality (classification)
            if series.nunique() < 10:
                score += 2

            # Reward numeric continuous (regression)
            if pd.api.types.is_numeric_dtype(series):
                score += 1

            # Penalize text-heavy columns
            if series.dtype == "object" and series.nunique() > 50:
                score -= 1

            # Reward typical target names
            if col.lower() in ["target", "label", "class", "churn", "y"]:
                score += 3

            scores[col] = score

        best_target = max(scores, key=scores.get)
        return best_target, scores

In [20]:
detector = TargetDetector()
target_col, score_map = detector.detect(df)

print("Suggested Target:", target_col)
print(score_map)


Suggested Target: Churn
{'customerID': -3, 'gender': 2, 'SeniorCitizen': 3, 'Partner': 2, 'Dependents': 2, 'tenure': 1, 'PhoneService': 2, 'MultipleLines': 2, 'InternetService': 2, 'OnlineSecurity': 2, 'OnlineBackup': 2, 'DeviceProtection': 2, 'TechSupport': 2, 'StreamingTV': 2, 'StreamingMovies': 2, 'Contract': 2, 'PaperlessBilling': 2, 'PaymentMethod': 2, 'MonthlyCharges': 1, 'TotalCharges': -1, 'Churn': 5}


In [21]:
X = df.drop(columns=[target_col])
y = df[target_col]


In [22]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns


In [23]:
def build_auto_pipeline(df, model):
    detector = TargetDetector()
    target_col, scores = detector.detect(df)

    print(f"Detected Target Column → {target_col}")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object", "category"]).columns

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ])

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    return pipeline, target_col


NameError: name 'scores' is not defined